# 01 — EEGNet (compact CNN baseline)

**Family:** CNN &nbsp;•&nbsp; **Input:** raw windowed signal `[14 x window_samples]`

EEGNet (Lawhern et al., 2018) is the lightweight reference point for the whole
benchmark. It factorizes EEG feature extraction into a temporal convolution
(frequency filters), a depthwise convolution across channels (spatial filters per
frequency band) and a separable convolution (feature-map mixing). It has the
smallest parameter budget of the eight models, which makes it the natural
accuracy-vs-efficiency anchor.

## 1. Setup

In [13]:
import os, sys, time, json
from functools import partial

import numpy as np
import torch

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.evaluation import (
    set_seed, get_device, count_parameters,
    make_default_fit, default_predict,
    run_subject_dependent, run_loso, save_results, quick_report,
)
from src.data_loader import (
    build_subject_dependent_loaders, build_loso_loaders, load_subject, REDUCED_5_CHANNELS,
)

set_seed(42)
DEVICE = get_device()

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
SD_SPLITS     = os.path.join(ROOT, "data", "splits", "subject_dependent_splits.json")
LOSO_SPLITS   = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR   = os.path.join(ROOT, "results")

# ----- experiment configuration -----
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> a 3-subject / 3-fold smoke test with few epochs
                      # False -> the full benchmark used for the thesis table
print("device:", DEVICE, "| label:", LABEL, "| QUICK:", QUICK)

from src.models import eegnet
MODEL_NAME = "EEGNet"

device: mps | label: valence | QUICK: False


## 2. Architecture

`src/models/eegnet.py` implements EEGNet-8,2 (F1 = 8 temporal filters, depth
multiplier D = 2, F2 = 16 separable filters, temporal kernel = 64 ≈ 0.5 s at
128 Hz). Depthwise kernels carry a max-norm constraint (via `torch.renorm`) as in
the reference implementation.

In [14]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, load_subject(PROCESSED_DIR, "subject_01").window_samples)}
_m = eegnet.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

EEGNet(
  (temporal): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (depthwise): Sequential(
    (0): _Constrained2d(8, 16, kernel_size=(14, 1), stride=(1, 1), groups=8, bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (4): Dropout(p=0.25, inplace=False)
  )
  (separable): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.25, inplace=False)
  )
  (classifier): Linear(in_feat

## Validation protocols

Both protocols are run through the shared harness in `src/evaluation.py` so the
numbers are directly comparable across all eight notebooks.

* **Subject-dependent** — for each of the 23 subjects a fresh model is trained on
  that subject's training split and tested on their held-out split
  (`data/splits/subject_dependent_splits.json`; whole film clips are kept on one
  side of the split to avoid window leakage). We report the mean ± std of the
  per-subject scores.
* **Subject-independent / LOSO** — 23 leave-one-subject-out folds
  (`data/splits/loso_splits.json`). Fold *k* trains on 22 subjects and tests on
  the 23rd; a 10% slice of the training subjects is used for early stopping.
  We report the mean ± std across the 23 folds.

The gap between the two is the central object of study in the thesis.

### 3. Subject-dependent protocol

In [15]:
FEATURE  = "raw"
CHANNELS = None
BATCH_SD = 64
EPOCHS_SD = 60
LR = 0.001

sd_loader = partial(
    build_subject_dependent_loaders,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_SD,
    channels=CHANNELS, normalize=True,
)
fit_sd = make_default_fit(epochs=(5 if QUICK else EPOCHS_SD), lr=LR)
sd_subjects = ["subject_01", "subject_02", "subject_03"] if QUICK else None

sd_res = run_subject_dependent(
    build_model=eegnet.build,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    loader_factory=sd_loader, fit_fn=fit_sd, predict_fn=default_predict,
    device=DEVICE, model_kwargs={"n_classes": 2, **{}},
    subjects=sd_subjects,
)
quick_report(MODEL_NAME, sd_res)

  [SD] subject_01 fold0: acc=0.480 f1=0.424 (11.9s)
  [SD] subject_02 fold0: acc=0.489 f1=0.377 (7.6s)
  [SD] subject_03 fold0: acc=0.687 f1=0.685 (8.2s)
  [SD] subject_04 fold0: acc=0.646 f1=0.646 (7.6s)
  [SD] subject_05 fold0: acc=0.276 f1=0.217 (7.3s)
  [SD] subject_06 fold0: acc=0.389 f1=0.280 (8.8s)
  [SD] subject_07 fold0: acc=0.667 f1=0.400 (8.3s)
  [SD] subject_08 fold0: acc=0.781 f1=0.711 (7.7s)
  [SD] subject_09 fold0: acc=0.449 f1=0.391 (7.1s)
  [SD] subject_10 fold0: acc=0.126 f1=0.112 (8.7s)
  [SD] subject_11 fold0: acc=0.607 f1=0.472 (7.5s)
  [SD] subject_12 fold0: acc=0.263 f1=0.213 (8.1s)
  [SD] subject_13 fold0: acc=0.638 f1=0.632 (9.0s)
  [SD] subject_14 fold0: acc=0.566 f1=0.554 (7.7s)
  [SD] subject_15 fold0: acc=0.275 f1=0.215 (7.7s)
  [SD] subject_16 fold0: acc=0.817 f1=0.450 (7.7s)
  [SD] subject_17 fold0: acc=0.713 f1=0.664 (7.4s)
  [SD] subject_18 fold0: acc=0.772 f1=0.436 (7.5s)
  [SD] subject_19 fold0: acc=0.509 f1=0.489 (7.0s)
  [SD] subject_20 fold0: acc=0

### 4. Subject-independent (LOSO) protocol

In [ ]:
BATCH_LOSO = 128
EPOCHS_LOSO = 50

loso_loader = partial(
    build_loso_loaders,
    processed_dir=PROCESSED_DIR, loso_path=LOSO_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_LOSO,
    channels=CHANNELS, normalize=True, val_frac=0.1,
)
fit_loso = make_default_fit(epochs=(5 if QUICK else EPOCHS_LOSO), lr=LR)
loso_folds = list(range(3)) if QUICK else None

loso_res = run_loso(
    build_model=eegnet.build, loso_path=LOSO_SPLITS,
    loader_factory=loso_loader, fit_fn=fit_loso, predict_fn=default_predict,
    device=DEVICE, model_kwargs={"n_classes": 2, **{}},
    folds=loso_folds,
)
quick_report(MODEL_NAME, loso_res)

  [LOSO] fold00 (subject_01): acc=0.620 f1=0.620 (148.8s)
  [LOSO] fold01 (subject_02): acc=0.569 f1=0.502 (155.5s)


## 5. Save results

In [ ]:
save_results(
    MODEL_NAME,
    subject_dependent=sd_res, loso=loso_res,
    family="CNN", results_dir=RESULTS_DIR,
    config={"label": LABEL, "feature": FEATURE, "channels": CHANNELS,
             "batch_sd": BATCH_SD, "batch_loso": BATCH_LOSO,
             "epochs_sd": EPOCHS_SD, "epochs_loso": EPOCHS_LOSO, "lr": LR,
             "quick": QUICK},
)

## Notes for the methodology chapter

* Every metric here comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* Parameter count and wall-clock training time are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` at the top to smoke-test the whole flow in a few minutes
  before committing to the full multi-hour run.
* `LABEL` switches the target between valence / arousal / dominance without any
  other change.